# Kernel Fusion Lab — GPU benchmark

Everything else in this project runs on CPU. This notebook is the one part that needs real hardware.

**Before running:** enable a GPU with `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

It takes a few minutes. At the end it prints a benchmark table and the full regenerated results document. Copy that output back into `docs/RESULTS.md` in the repo, or download it from the file browser on the left.

## 1. Confirm a GPU is actually attached

If this cell says no GPU, stop and change the runtime type. Running without one produces nothing useful — the harness will correctly refuse to report timings.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU attached. Runtime -> Change runtime type -> T4 GPU, then Run all again.'
    )

print('GPU        ', torch.cuda.get_device_name(0))
print('torch      ', torch.__version__)

import triton
print('triton     ', triton.__version__)

props = torch.cuda.get_device_properties(0)
print('SMs        ', props.multi_processor_count)
print('total mem  ', f'{props.total_memory / 1024**3:.1f} GiB')

## 2. Fetch the project

In [ ]:
import os

REPO = 'https://github.com/VedangDugar/kernel-fusion-lab.git'

if not os.path.isdir('kernel-fusion-lab'):
    !git clone --depth 1 $REPO

%cd kernel-fusion-lab
!ls

## 3. Correctness first

The CPU interpreter already validated these kernels, but the interpreter is a NumPy simulation. This re-runs the same suite against real compiled PTX, which is the version that actually matters. Benchmarking a kernel before confirming it is correct on the hardware you are timing would be meaningless.

In [ ]:
!pip install -q tabulate
!python -m pytest -q

## 4. Peak bandwidth

Measured with a large device-to-device copy rather than taken from the datasheet. No real kernel reaches the datasheet figure, so using it as the denominator would understate how close these kernels get to the roofline.

In [ ]:
import sys
sys.path.insert(0, '.')

from harness import benchmark

peak = benchmark.measure_peak_bandwidth()
print(f'achievable bandwidth: {peak:.1f} GB/s')

## 5. Benchmark all four providers

- `triton_fused` — the one-pass fused kernel
- `triton_unfused` — the same work as two kernels, with the intermediate round-tripping through HBM
- `torch_eager` — naive PyTorch, every intermediate materialised
- `torch_compile` — PyTorch Inductor, which fuses these operations and emits Triton itself

`torch_compile` is the hard baseline. Parity with it is a good result and beating it is not expected; the interesting comparison for the fusion argument is `triton_fused` against `triton_unfused`.

In [ ]:
from harness.config import DTYPES, SHAPES

timings = benchmark.run_all(SHAPES, DTYPES)
print(benchmark.format_markdown(timings, peak))

## 6. Does the measurement match the prediction?

The analytical model predicts the fused kernel moves half the bytes of the unfused pair. If both variants run near the bandwidth roofline, the measured speedup should approach 2x. If it falls well short, the kernels are not bandwidth bound and the model, while still arithmetically correct, does not predict the runtime.

In [ ]:
from harness import memory_model
from harness.config import DTYPE_NAMES

by_key = {(t.shape, t.dtype, t.provider): t for t in timings}

print(f"{'shape':>12} {'dtype':>9} {'predicted':>10} {'measured':>9} {'gap':>8}")
print('-' * 52)
for dtype in DTYPES:
    for shape in SHAPES:
        name = DTYPE_NAMES[dtype]
        fused = by_key.get((shape, name, 'triton_fused'))
        unfused = by_key.get((shape, name, 'triton_unfused'))
        if not (fused and unfused and fused.measured and unfused.measured):
            continue
        m = memory_model.model(shape, dtype)
        predicted = m.unfused_bytes / m.fused_bytes
        measured = unfused.ms_median / fused.ms_median
        print(
            f'{str(shape):>12} {name:>9} {predicted:>9.2f}x {measured:>8.2f}x '
            f'{measured / predicted:>7.0%}'
        )

## 7. Regenerate the full results document

This overwrites `docs/RESULTS.md` with GPU numbers in place of the "not measured" fields. Copy the printed output into the repo.

In [ ]:
!python -m harness.sweep
print()
print(open('docs/RESULTS.md').read())